In [13]:
from google.colab import files
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, pipeline
import torch
from sentence_transformers import SentenceTransformer, util
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### Introduction to Hugging Face transformers and datas

Question 1 - Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here.

In [2]:
upload = files.upload()
dataset = load_dataset("csv", data_files="train.csv")["train"]

Saving train.csv to train.csv


Generating train split: 0 examples [00:00, ? examples/s]

In [3]:
def combine(example):
    return {
        "combined_text": example["prompt"] + " " + example["A"]
    }

dataset = dataset.map(combine)
result = len(dataset[51]["combined_text"])
result

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

614

Question 2 - Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?

In [4]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
vocab_size = tokenizer.vocab_size
vocab_size

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

30522

Question 3 - Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.

In [5]:
sep_id = tokenizer.convert_tokens_to_ids("[SEP]")
sep_id

102

Question 4 - Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors).

What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

In [6]:
prompts = [str(x) if x is not None else "" for x in dataset["prompt"]]

encodings = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

encodings["input_ids"].shape

torch.Size([2000, 128])

### BERT/RoBERTa Architecture & Attention Mechanisms

Question 5 - A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer.

In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?

In [7]:
hidden_size = 768
num_heads = 12

head_dim = hidden_size // num_heads
print(head_dim)

64


Question 6 - Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object.

What is the exact shape of the last_hidden_state tensor returned?

Note: We follow zero-indexing here.

In [8]:
model = AutoModel.from_pretrained("bert-base-uncased")

text = str(dataset[0]["prompt"])
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
outputs.last_hidden_state.shape

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 31, 768])

Question 7 -   Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).  

In [9]:
cls_vector = outputs.last_hidden_state[0, 0, :]
result = cls_vector[:5].sum().item()
result

-1.2000958919525146

Question 8 - Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0).

What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).

In [10]:
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    attn_implementation="eager"
)

text = "Light-ion fusion is a technique."
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
fusion_index = tokens.index("fusion")
attn = outputs.attentions[-1]

value = attn[0, 0, 0, fusion_index].item()
value

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.10247292369604111

### Context-Aware Embeddings

Question 9 - Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.

In [11]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

prompt = str(dataset[0]["prompt"])
option_b = str(dataset[0]["B"])

emb_prompt = model.encode(prompt, convert_to_tensor=True)
emb_b = model.encode(option_b, convert_to_tensor=True)

score = util.cos_sim(emb_prompt, emb_b)
result = score.item()
result

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

0.7658097743988037

Question 10 - Build two complete ranking pipelines evaluating every row in train.csv.

Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set?

Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?  

In [12]:
def tfidf_top3(example):
    texts = [example["prompt"]] + [example[c] for c in ["A","B","C","D","E"]]
    vectorizer = TfidfVectorizer()
    tfidf = vectorizer.fit_transform(texts)
    sims = cosine_similarity(tfidf[0:1], tfidf[1:]).flatten()
    top3_idx = sims.argsort()[-3:][::-1]
    options = ["A","B","C","D","E"]
    return [options[i] for i in top3_idx]

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def minilm_top3(example):
    prompt_emb = model.encode(example["prompt"], convert_to_tensor=True)
    option_texts = [example[c] for c in ["A","B","C","D","E"]]
    option_embs = model.encode(option_texts, convert_to_tensor=True)
    sims = util.cos_sim(prompt_emb, option_embs)[0]
    top3_idx = sims.argsort(descending=True)[:3]
    options = ["A","B","C","D","E"]
    return [options[i] for i in top3_idx]

tfidf_preds = []
minilm_preds = []
actuals = []

for ex in dataset:
    actual = ex["answer"]
    tfidf_p = tfidf_top3(ex)
    minilm_p = minilm_top3(ex)
    tfidf_preds.append(tfidf_p)
    minilm_preds.append(minilm_p)
    actuals.append(actual)

count = 0
for a, tfidf_p, minilm_p in zip(actuals, tfidf_preds, minilm_preds):
    if (a not in tfidf_p) and (a in minilm_p):
        count += 1
count

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

564

### Zero-shot classification concepts

Question 11 - Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places).

In [14]:
classifier = pipeline("zero-shot-classification")

row = dataset[1]
prompt = row["prompt"]
candidate_labels = [row["A"], row["B"], row["C"]]

result = classifier(prompt, candidate_labels)
top_score = result["scores"][0]
top_score

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

0.45745277404785156

Question 12 - Run the exact same zero-shot classification as the previous question, but this time pass the argument multi_label=True.

What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?

In [15]:
res_softmax = classifier(prompt, candidate_labels, multi_label=False)
sum_softmax = sum(res_softmax["scores"])

res_sigmoid = classifier(prompt, candidate_labels, multi_label=True)
sum_sigmoid = sum(res_sigmoid["scores"])

diff = abs(sum_softmax - sum_sigmoid)
diff

0.9994903945043916

Question 13 - Load a Small Language Model like google/flan-t5-small using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: "Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B."
Pass this string to the pipeline, setting max_new_tokens=5. What is the exact string output returned by the model?

In [17]:
row = dataset[0]
prompt = row["prompt"]
A = row["A"]
B = row["B"]
input_text = f"Question: {prompt}. Is the correct answer A: {A} or B: {B}? Answer with just the letter A or B."

generator = pipeline("text-generation", model="google/flan-t5-small")
result = generator(input_text, max_new_tokens=5)
result[0]["generated_text"]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCaus

"Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B."